# Sequence-to-Sequence Models (Encoder–Decoder)

> Based on Stanford CS230 — C5M3, Andrew Ng / DeepLearning.AI

Machine translation, summarisation, and speech-to-text all require mapping a variable-length input sequence to a variable-length output sequence. The **encoder–decoder** (seq2seq) architecture handles this by compressing the input into a fixed context vector, then generating the output token by token.

## Learning Objectives

1. Explain the encoder–decoder architecture and the role of the context vector
2. Distinguish conditional language modelling from unconditional language modelling
3. Describe the training objective (teacher forcing) and inference (autoregressive decoding)
4. Understand why attention was introduced to overcome the bottleneck context vector

## Architecture

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 700 190" width="700" height="190" style="font-family:DejaVu Sans,Arial,sans-serif;background:#f5f5f7;border-radius:8px">
  <defs>
    <marker id="sa" markerWidth="7" markerHeight="7" refX="5" refY="3" orient="auto">
      <path d="M0,0 L0,6 L7,3 z" fill="#888"/>
    </marker>
  </defs>
  <!-- ENCODER label -->
  <text x="130" y="18" text-anchor="middle" fill="#b03a3a" font-size="11" font-weight="bold">ENCODER</text>
  <!-- DECODER label -->
  <text x="520" y="18" text-anchor="middle" fill="#8a3ab8" font-size="11" font-weight="bold">DECODER</text>
  <!-- Encoder cells: 3 -->
  <rect x="20"  y="70" width="60" height="50" rx="5" fill="#e05c5c" fill-opacity="0.15" stroke="#e05c5c" stroke-width="1.6"/>
  <text x="50"  y="99" text-anchor="middle" fill="#b03a3a" font-size="10">RNN₁</text>
  <rect x="110" y="70" width="60" height="50" rx="5" fill="#e05c5c" fill-opacity="0.15" stroke="#e05c5c" stroke-width="1.6"/>
  <text x="140" y="99" text-anchor="middle" fill="#b03a3a" font-size="10">RNN₂</text>
  <rect x="200" y="70" width="60" height="50" rx="5" fill="#e05c5c" fill-opacity="0.15" stroke="#e05c5c" stroke-width="1.6"/>
  <text x="230" y="99" text-anchor="middle" fill="#b03a3a" font-size="10">RNN₃</text>
  <!-- Encoder inputs -->
  <line x1="50"  y1="140" x2="50"  y2="120" stroke="#aaa" stroke-width="1.2" marker-end="url(#sa)"/>
  <line x1="140" y1="140" x2="140" y2="120" stroke="#aaa" stroke-width="1.2" marker-end="url(#sa)"/>
  <line x1="230" y1="140" x2="230" y2="120" stroke="#aaa" stroke-width="1.2" marker-end="url(#sa)"/>
  <text x="50"  y="155" text-anchor="middle" fill="#b03a3a" font-size="10">Je</text>
  <text x="140" y="155" text-anchor="middle" fill="#b03a3a" font-size="10">suis</text>
  <text x="230" y="155" text-anchor="middle" fill="#b03a3a" font-size="10">étudiant</text>
  <!-- Encoder hidden connections -->
  <line x1="80"  y1="95" x2="110" y2="95" stroke="#e05c5c" stroke-width="1.4" marker-end="url(#sa)"/>
  <line x1="170" y1="95" x2="200" y2="95" stroke="#e05c5c" stroke-width="1.4" marker-end="url(#sa)"/>
  <!-- Context vector -->
  <rect x="295" y="75" width="80" height="40" rx="5" fill="#f4b942" fill-opacity="0.20" stroke="#f4b942" stroke-width="1.8"/>
  <text x="335" y="97" text-anchor="middle" fill="#a07010" font-size="10" font-weight="bold">context</text>
  <text x="335" y="110" text-anchor="middle" fill="#a07010" font-size="9">a⁽ᵀˣ⁾</text>
  <line x1="260" y1="95" x2="295" y2="95" stroke="#f4b942" stroke-width="1.8" marker-end="url(#sa)"/>
  <!-- Decoder cells: 3 -->
  <rect x="395" y="70" width="60" height="50" rx="5" fill="#c678dd" fill-opacity="0.15" stroke="#c678dd" stroke-width="1.6"/>
  <text x="425" y="99" text-anchor="middle" fill="#8a3ab8" font-size="10">RNN₁</text>
  <rect x="485" y="70" width="60" height="50" rx="5" fill="#c678dd" fill-opacity="0.15" stroke="#c678dd" stroke-width="1.6"/>
  <text x="515" y="99" text-anchor="middle" fill="#8a3ab8" font-size="10">RNN₂</text>
  <rect x="575" y="70" width="60" height="50" rx="5" fill="#c678dd" fill-opacity="0.15" stroke="#c678dd" stroke-width="1.6"/>
  <text x="605" y="99" text-anchor="middle" fill="#8a3ab8" font-size="10">RNN₃</text>
  <!-- Context → Decoder -->
  <line x1="375" y1="95" x2="395" y2="95" stroke="#f4b942" stroke-width="1.8" marker-end="url(#sa)"/>
  <!-- Decoder hidden connections -->
  <line x1="455" y1="95" x2="485" y2="95" stroke="#c678dd" stroke-width="1.4" marker-end="url(#sa)"/>
  <line x1="545" y1="95" x2="575" y2="95" stroke="#c678dd" stroke-width="1.4" marker-end="url(#sa)"/>
  <!-- Decoder outputs -->
  <line x1="425" y1="70" x2="425" y2="50" stroke="#aaa" stroke-width="1.2" marker-end="url(#sa)"/>
  <line x1="515" y1="70" x2="515" y2="50" stroke="#aaa" stroke-width="1.2" marker-end="url(#sa)"/>
  <line x1="605" y1="70" x2="605" y2="50" stroke="#aaa" stroke-width="1.2" marker-end="url(#sa)"/>
  <text x="425" y="44" text-anchor="middle" fill="#8a3ab8" font-size="10">I</text>
  <text x="515" y="44" text-anchor="middle" fill="#8a3ab8" font-size="10">am</text>
  <text x="605" y="44" text-anchor="middle" fill="#8a3ab8" font-size="10">a student</text>
  <!-- Teacher forcing note -->
  <text x="350" y="180" text-anchor="middle" fill="#888" font-size="9" font-style="italic">Training: feed ground-truth tokens as decoder input (teacher forcing)</text>
</svg>

## Conditional Language Model

At inference, the decoder samples the most probable output sequence:

$$P(y^{\langle 1 \rangle}, \ldots, y^{\langle T_y \rangle} \mid x^{\langle 1 \rangle}, \ldots, x^{\langle T_x \rangle}) = \prod_{t=1}^{T_y} P\!\bigl(y^{\langle t \rangle} \mid x, y^{\langle 1 \rangle}, \ldots, y^{\langle t-1 \rangle}\bigr)$$

Greedy decoding picks $\arg\max$ at each step; beam search maintains the $B$ highest-probability partial sequences.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#f5f5f7', 'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#c8ccd4', 'axes.labelcolor': '#1a1d27',
    'axes.titlecolor': '#1a1d27', 'xtick.color': '#2a2e3a',
    'ytick.color': '#2a2e3a', 'grid.color': '#e0e3ea',
    'grid.linestyle': '--', 'grid.alpha': 0.5,
    'text.color': '#1a1d27', 'font.family': 'DejaVu Sans',
    'axes.titlesize': 12, 'axes.labelsize': 11,
    'legend.facecolor': '#ffffff', 'legend.edgecolor': '#c8ccd4',
    'figure.dpi': 110,
})
P = ['#5b9bd5', '#e05c5c', '#f4b942', '#7ecba1', '#56b6c2', '#c678dd']

# ---- Minimal seq2seq forward pass (illustrative) ----
sigmoid = lambda z: 1 / (1 + np.exp(-z))

def softmax(z):
    e = np.exp(z - z.max())
    return e / e.sum()

np.random.seed(42)

class SimpleSeq2Seq:
    """Toy encoder-decoder for shape illustration only."""
    def __init__(self, n_x, n_y, n_a, seed=0):
        np.random.seed(seed)
        s = 0.05
        # encoder weights (shared)
        self.Wax = np.random.randn(n_a, n_x) * s
        self.Waa = np.random.randn(n_a, n_a) * s
        self.ba  = np.zeros((n_a, 1))
        # decoder weights
        self.Wda = np.random.randn(n_a, n_a + n_y) * s
        self.bd  = np.zeros((n_a, 1))
        self.Wya = np.random.randn(n_y, n_a) * s
        self.by  = np.zeros((n_y, 1))

    def encode(self, x_seq):
        a = np.zeros((self.Waa.shape[0], 1))
        for x in x_seq:
            a = np.tanh(self.Waa @ a + self.Wax @ x + self.ba)
        return a  # context vector

    def decode_step(self, y_prev, a_prev):
        z = np.vstack([a_prev, y_prev])
        a = np.tanh(self.Wda @ z + self.bd)
        y = softmax(self.Wya @ a + self.by)
        return a, y

n_x, n_y, n_a = 6, 5, 12
model = SimpleSeq2Seq(n_x, n_y, n_a)
T_x = 4
x_seq = [np.random.randn(n_x, 1) for _ in range(T_x)]
context = model.encode(x_seq)

T_y = 4
outputs = []
a = context
y = np.zeros((n_y, 1))   # <SOS> token
for _ in range(T_y):
    a, y = model.decode_step(y, a)
    outputs.append(y.flatten())

outputs = np.array(outputs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Seq2Seq Encoder–Decoder — Illustration', fontsize=13, fontweight='bold')

# Decoder output probability distributions over time
im = axes[0].imshow(outputs.T, aspect='auto', cmap='Blues', vmin=0, vmax=0.5)
axes[0].set_xlabel('Decoder time step')
axes[0].set_ylabel('Output token class')
axes[0].set_title('Output distribution $P(y^{\\langle t \\rangle}|x,y^{<t})$')
axes[0].set_xticks(range(T_y))
axes[0].set_yticks(range(n_y))
axes[0].set_xticklabels([f't={i+1}' for i in range(T_y)])
plt.colorbar(im, ax=axes[0], label='probability')
for i in range(T_y):
    for j in range(n_y):
        axes[0].text(i, j, f'{outputs[i,j]:.2f}', ha='center', va='center', fontsize=8)

# Greedy vs random comparison
np.random.seed(10)
greedy_tokens = [o.argmax() for o in outputs]
random_tokens = [np.random.choice(n_y, p=o) for o in outputs]
token_labels   = [f'tok{i}' for i in range(n_y)]
x_pos = np.arange(T_y)
axes[1].plot(x_pos, greedy_tokens, 'o-', color=P[3], lw=2, label='Greedy (argmax)', ms=8)
axes[1].plot(x_pos, random_tokens, 's--', color=P[1], lw=2, label='Random sample', ms=8)
axes[1].set_xticks(x_pos); axes[1].set_xticklabels([f't={i+1}' for i in range(T_y)])
axes[1].set_yticks(range(n_y)); axes[1].set_yticklabels(token_labels)
axes[1].set_title('Greedy vs Sampled Decoding'); axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f"Context vector shape:  {context.shape}")
print(f"Greedy decoded tokens: {greedy_tokens}")
